# Swin Transformer — semantic segmentation (BRISC)

**Swin** (hierarchical vision transformer with shifted windows) as encoder via **[timm](https://github.com/huggingface/pytorch-image-models)**, plus a lightweight **multi-scale fusion decoder** (no U-Net skips — fused pyramid to logits).

**Paper:** [Swin Transformer: Hierarchical Vision Transformer using Shifted Windows](https://arxiv.org/abs/2103.14030)

**Dependency:** `pip install timm` (see `requirements.txt`). Model definition: **`swin_seg.py`**.

In [1]:
import os
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for _d in [
    _cwd,
    _cwd / "Segmentation_Transformers",
    _cwd / "BRISC" / "Segmentation_Transformers",
]:
    if (_d / "swin_seg.py").is_file():
        sys.path.insert(0, str(_d))
        break

import math
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from swin_seg import SwinSeg


def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

/home/shaunakmishra25/Desktop/ml_env/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============== CONFIG ==============
IMG_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 40
LR = 1e-4
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.1
MIN_LR_RATIO = 0.01
MAX_GRAD_NORM = 1.0
NUM_CLASSES = 1
DICE_LAMBDA = 0.5

# Swin: 'tiny', 'small', or 'base' (ImageNet pretrained backbone when PRETRAINED_BACKBONE=True)
MODEL_VARIANT = "tiny"
PRETRAINED_BACKBONE = True

BASE_PATH = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task"
TRAIN_IMG = os.path.join(BASE_PATH, "train", "images")
TRAIN_MASK = os.path.join(BASE_PATH, "train", "masks")
TEST_IMG = os.path.join(BASE_PATH, "test", "images")
TEST_MASK = os.path.join(BASE_PATH, "test", "masks")

PRED_BASE = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/SwinSeg"
PRED_MASK_DIR = os.path.join(PRED_BASE, "masks")
OVERLAY_DIR = os.path.join(PRED_BASE, "overlays")
MODEL_SAVE_DIR = os.path.join(os.path.dirname(PRED_BASE), "models")
_ptag = f"{MODEL_VARIANT}_pre" if PRETRAINED_BACKBONE else f"{MODEL_VARIANT}_scratch"
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, f"swinseg_{_ptag}.pth")
BEST_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, f"swinseg_{_ptag}_best.pth")

os.makedirs(PRED_MASK_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

## 1. Dataset (ImageNet norm + train augmentation)

Use **IMG_SIZE divisible by 32** (256 is fine for Swin).

In [3]:
IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]


class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, augment=False):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.masks = sorted(os.listdir(mask_dir))
        self.augment = augment
        self.normalize = transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD)
        self.color_jitter = transforms.ColorJitter(
            brightness=0.25, contrast=0.25, saturation=0.2, hue=0.05
        )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        image = image.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        mask = mask.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)

        if self.augment:
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)
                mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
            if random.random() < 0.9:
                image = self.color_jitter(image)

        image = transforms.ToTensor()(image)
        image = self.normalize(image)
        mask = np.array(mask).astype("float32") / 255.0
        mask = (mask > 0.5).astype("float32")
        mask = torch.tensor(mask)
        return image, mask, self.masks[idx]


def collate_fn(batch):
    return (
        torch.stack([b[0] for b in batch]),
        torch.stack([b[1] for b in batch]),
        [b[2] for b in batch],
    )

## 2. Model + DataLoader

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SwinSeg(
    num_classes=NUM_CLASSES,
    variant=MODEL_VARIANT,
    pretrained=PRETRAINED_BACKBONE,
    img_size=IMG_SIZE,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

train_dataset = SegDataset(TRAIN_IMG, TRAIN_MASK, augment=True)
test_dataset = SegDataset(TEST_IMG, TEST_MASK, augment=False)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

print(f"Device: {device}")
print(f"Swin variant: {MODEL_VARIANT}, pretrained backbone: {PRETRAINED_BACKBONE}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

Device: cuda
Swin variant: tiny, pretrained backbone: True
Params: 30,838,907


## 3. Train (BCE + Dice, warmup + cosine)

In [5]:
steps_per_epoch = len(train_loader)
total_steps = max(1, EPOCHS * steps_per_epoch)
warmup_steps = max(1, int(WARMUP_RATIO * total_steps))


def _lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)
bce_loss = nn.BCEWithLogitsLoss()


def dice_loss_logits(logits, targets, smooth=1e-6):
    p = torch.sigmoid(logits).view(logits.shape[0], -1)
    t = targets.view(targets.shape[0], -1)
    inter = (p * t).sum(dim=1)
    union = p.sum(dim=1) + t.sum(dim=1)
    dice = (2 * inter + smooth) / (union + smooth)
    return (1 - dice).mean()


def seg_loss(logits, masks):
    return bce_loss(logits, masks) + DICE_LAMBDA * dice_loss_logits(logits, masks)


def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for images, masks, _ in loader:
        images = images.to(device)
        masks = masks.unsqueeze(1).to(device)
        optimizer.zero_grad()
        logits = model(images)
        logits = F.interpolate(logits, size=masks.shape[2:], mode="bilinear", align_corners=False)
        loss = seg_loss(logits, masks)
        loss.backward()
        if MAX_GRAD_NORM > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(model, loader, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, masks, _ in loader:
            images = images.to(device)
            masks = masks.unsqueeze(1).to(device)
            logits = model(images)
            logits = F.interpolate(logits, size=masks.shape[2:], mode="bilinear", align_corners=False)
            total_loss += seg_loss(logits, masks).item()
    return total_loss / len(loader)

In [6]:
best_val_loss = float("inf")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss = validate(model, test_loader, device)
    lr_now = optimizer.param_groups[0]["lr"]
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    print(
        f"Epoch [{epoch}/{EPOCHS}] Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} LR: {lr_now:.2e}"
    )

Epoch [1/40] Loss: 0.6806 Val Loss: 0.4499 LR: 2.50e-05
Epoch [2/40] Loss: 0.3188 Val Loss: 0.1838 LR: 5.00e-05
Epoch [3/40] Loss: 0.1480 Val Loss: 0.1317 LR: 7.50e-05
Epoch [4/40] Loss: 0.1155 Val Loss: 0.1114 LR: 1.00e-04
Epoch [5/40] Loss: 0.1053 Val Loss: 0.1046 LR: 9.98e-05
Epoch [6/40] Loss: 0.0964 Val Loss: 0.0983 LR: 9.92e-05
Epoch [7/40] Loss: 0.0905 Val Loss: 0.0937 LR: 9.83e-05
Epoch [8/40] Loss: 0.0876 Val Loss: 0.0979 LR: 9.70e-05
Epoch [9/40] Loss: 0.0827 Val Loss: 0.0876 LR: 9.54e-05
Epoch [10/40] Loss: 0.0786 Val Loss: 0.0910 LR: 9.34e-05
Epoch [11/40] Loss: 0.0772 Val Loss: 0.0933 LR: 9.10e-05
Epoch [12/40] Loss: 0.0743 Val Loss: 0.0906 LR: 8.84e-05
Epoch [13/40] Loss: 0.0706 Val Loss: 0.0896 LR: 8.55e-05
Epoch [14/40] Loss: 0.0683 Val Loss: 0.0878 LR: 8.23e-05
Epoch [15/40] Loss: 0.0658 Val Loss: 0.0857 LR: 7.89e-05
Epoch [16/40] Loss: 0.0656 Val Loss: 0.0869 LR: 7.52e-05
Epoch [17/40] Loss: 0.0627 Val Loss: 0.0826 LR: 7.14e-05
Epoch [18/40] Loss: 0.0607 Val Loss: 0.0

In [8]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Saved: {MODEL_SAVE_PATH}")
print(f"Best val: {BEST_MODEL_PATH}")

Saved: /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/models/swinseg_tiny_pre.pth
Best val: /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/models/swinseg_tiny_pre_best.pth


## 4. Inference

Saves masks + overlays under `PRED_BASE`.

In [9]:
def denormalize(tensor):
    mean = torch.tensor(IMAGE_MEAN, device=tensor.device, dtype=tensor.dtype).view(1, 3, 1, 1)
    std = torch.tensor(IMAGE_STD, device=tensor.device, dtype=tensor.dtype).view(1, 3, 1, 1)
    return tensor * std + mean


model.eval()
with torch.no_grad():
    for images, masks, names in test_loader:
        images = images.to(device)
        logits = model(images)
        logits = F.interpolate(logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        preds = (torch.sigmoid(logits).cpu().numpy() > 0.5).astype(np.uint8)
        for i, name in enumerate(names):
            pm = preds[i, 0]
            Image.fromarray((pm * 255).astype(np.uint8)).save(
                os.path.join(PRED_MASK_DIR, name)
            )
            vis = denormalize(images[i : i + 1]).squeeze(0).permute(1, 2, 0).cpu().numpy()
            vis = np.clip(vis, 0, 1)
            overlay = vis.copy()
            overlay[pm > 0] = [1, 0, 0]
            blended = (np.clip(0.7 * vis + 0.3 * overlay, 0, 1) * 255).astype(np.uint8)
            Image.fromarray(blended).save(os.path.join(OVERLAY_DIR, name))
print(f"Done: {len(test_dataset)} samples → {PRED_BASE}")

Done: 860 samples → /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/SwinSeg


## 5. Load best checkpoint (optional)

Use this if you **restarted the kernel** or want metrics/inference from **`BEST_MODEL_PATH`** instead of the last epoch in memory. Run **before** re-running the inference cell.

In [10]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()
print("Loaded:", BEST_MODEL_PATH)

Loaded: /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/models/swinseg_tiny_pre_best.pth


## 6. Test metrics

Requires saved masks under `PRED_BASE/masks/` (run **§4 Inference** first).

In [11]:
def dice_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    inter = np.sum(y_true * y_pred)
    return (2.0 * inter + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)


def iou_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    inter = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - inter
    return (inter + smooth) / (union + smooth)


def precision_recall_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return tp / (tp + fp + 1e-6), tp / (tp + fn + 1e-6)


def accuracy_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    return np.sum(y_true == y_pred) / len(y_true)


dice_l, iou_l, p_l, r_l, a_l = [], [], [], [], []
for i in range(len(test_dataset)):
    _, mask, name = test_dataset[i]
    gt = (mask.numpy() > 0.5).astype(np.uint8)
    pred_img = np.array(Image.open(os.path.join(PRED_MASK_DIR, name)))
    pr = (pred_img > 127).astype(np.uint8)
    dice_l.append(dice_np(gt, pr))
    iou_l.append(iou_np(gt, pr))
    p, rr = precision_recall_np(gt, pr)
    p_l.append(p)
    r_l.append(rr)
    a_l.append(accuracy_np(gt, pr))

print("===== TEST SET RESULTS =====")
print(f"Mean Dice     : {np.mean(dice_l):.4f}")
print(f"Mean IoU      : {np.mean(iou_l):.4f}")
print(f"Mean Precision: {np.mean(p_l):.4f}")
print(f"Mean Recall   : {np.mean(r_l):.4f}")
print(f"Mean Accuracy : {np.mean(a_l):.4f}")

===== TEST SET RESULTS =====
Mean Dice     : 0.8765
Mean IoU      : 0.8088
Mean Precision: 0.8828
Mean Recall   : 0.8936
Mean Accuracy : 0.9965


## 7. Comparison PDF (Image, GT, Pred, Overlay, TP/FP/FN)

Run after inference so `masks/` and `overlays/` exist. Writes `comparison_report.pdf` under `PRED_BASE`.

In [12]:
SAMPLES_PER_PAGE = 4
COMPARISON_PDF_PATH = os.path.join(PRED_BASE, "comparison_report.pdf")


def make_comparison_overlay(gt_mask, pred_mask):
    gt = (gt_mask > 0.5).astype(np.uint8)
    pred = (pred_mask > 0.5).astype(np.uint8)
    h, w = gt.shape
    o = np.zeros((h, w, 3), dtype=np.uint8)
    o[gt == 1] = [0, 0, 128]
    o[pred == 1] = [128, 0, 0]
    o[(gt == 1) & (pred == 1)] = [0, 128, 0]
    return o


indices = np.arange(len(test_dataset))
with PdfPages(COMPARISON_PDF_PATH) as pdf:
    for page_start in range(0, len(indices), SAMPLES_PER_PAGE):
        page_indices = indices[page_start : page_start + SAMPLES_PER_PAGE]
        n_rows = len(page_indices)
        fig, axes = plt.subplots(n_rows, 5, figsize=(15, 3 * n_rows))
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        for row, idx in enumerate(page_indices):
            img_name = test_dataset.images[idx]
            mask_name = test_dataset.masks[idx]
            img_path = os.path.join(TEST_IMG, img_name)
            gt_path = os.path.join(TEST_MASK, mask_name)
            pred_path = os.path.join(PRED_MASK_DIR, mask_name)
            overlay_path = os.path.join(OVERLAY_DIR, mask_name)
            image = np.array(Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            gt_mask = np.array(Image.open(gt_path).convert("L").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            pred_mask = np.array(Image.open(pred_path).convert("L")) / 255.0
            if pred_mask.shape != (IMG_SIZE, IMG_SIZE):
                pred_mask = np.array(
                    Image.fromarray((pred_mask * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))
                ) / 255.0
            overlay_img = np.array(Image.open(overlay_path).convert("RGB")) / 255.0
            if overlay_img.shape[:2] != (IMG_SIZE, IMG_SIZE):
                overlay_img = np.array(
                    Image.fromarray((overlay_img * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))
                ) / 255.0
            comparison = make_comparison_overlay(gt_mask, pred_mask)
            axes[row, 0].imshow(image)
            axes[row, 0].set_title("Image")
            axes[row, 0].axis("off")
            axes[row, 1].imshow(gt_mask, cmap="gray")
            axes[row, 1].set_title("Ground Truth")
            axes[row, 1].axis("off")
            axes[row, 2].imshow(pred_mask, cmap="gray")
            axes[row, 2].set_title("Predicted")
            axes[row, 2].axis("off")
            axes[row, 3].imshow(overlay_img)
            axes[row, 3].set_title("Overlay")
            axes[row, 3].axis("off")
            axes[row, 4].imshow(comparison)
            axes[row, 4].set_title("TP=green, FP=red, FN=blue")
            axes[row, 4].axis("off")
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Comparison PDF saved to {COMPARISON_PDF_PATH}")

Comparison PDF saved to /run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/SwinSeg/comparison_report.pdf
